In [ ]:
import json
import pandas as pd

with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())
print(df["sentiment"].value_counts())

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../"))

In [ ]:
import re, html

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#', '', text)
    # text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # text = text.replace('\n', ' ')
    # text = re.sub(r'\d+', '', text)
    # text = re.sub(r'\s+', ' ', text).strip()
    return text.strip()

df["text"] = df["text"].apply(clean_text)

def has_thai(text):
    return bool(re.search(r'[ก-๙]', text))

df = df[df["text"].apply(has_thai)]

text = df["text"]
sentiment = df["sentiment"]

In [ ]:
from sklearn.model_selection import train_test_split

text_train, text_val, sentiment_train, sentiment_val = train_test_split(
    text, sentiment, test_size=0.2, random_state=42
)

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from pythainlp.tokenize import word_tokenize
# from models.preprocess import tokenizer

# # def tokenize(text):
# #     return word_tokenize(text, engine="newmm")

# vectorizer = TfidfVectorizer(
#     tokenizer=tokenizer,
#     ngram_range=(1,2),
# )

# text_train_tfidf = vectorizer.fit_transform(text_train)
# text_val_tfidf = vectorizer.transform(text_val)

In [ ]:
# from sklearn.svm import LinearSVC
# from sklearn.metrics import classification_report, accuracy_score


# model = LinearSVC(C=1.5)
# model.fit(text_train_tfidf, sentiment_train)

# y_pred = model.predict(text_val_tfidf)
# print(classification_report(sentiment_val, y_pred))
# print("Accuracy:", accuracy_score(sentiment_val, y_pred))

In [ ]:
# import pickle

# with open("sentiment_model_new.pkl", "wb") as f:
#     pickle.dump((vectorizer, model), f)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

from models.preprocess import tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion


word_vectorizer = TfidfVectorizer(
    tokenizer=tokenizer,
    ngram_range=(1,3),
    sublinear_tf=True,
    max_df=0.9,
    min_df=2
)


# TF-IDF CHAR

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,6),
    sublinear_tf=True,
    max_df=0.9,
    min_df=3
)


# COMBINE FEATURES

combined_features = FeatureUnion(
    [
        ("word_tfidf", word_vectorizer),
        ("char_tfidf", char_vectorizer)
    ],
    transformer_weights={
        "word_tfidf": 1.0,
        "char_tfidf": 0.7
    }
)


# BASE MODELS

svm = LinearSVC(
    C=2,
    max_iter=5000
)

logreg = LogisticRegression(
    max_iter=3000
)

nb = MultinomialNB()

rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1
)

knn = KNeighborsClassifier(
    n_neighbors=5
)


# STACKING ENSEMBLE

stack_model = StackingClassifier(

    estimators=[
        ("svm", svm),
        ("lr", logreg),
        ("nb", nb),
        ("rf", rf),
        ("knn", knn)
    ],

    final_estimator=LogisticRegression(),

    n_jobs=-1
)


# PIPELINE

pipeline = Pipeline([

    ("features", combined_features),

    ("classifier", stack_model)

])


# TRAIN

print("Training Advanced Sentiment Model...")

pipeline.fit(text_train, sentiment_train)


# PREDICT

y_pred = pipeline.predict(text_val)

print("Accuracy:", accuracy_score(sentiment_val, y_pred))

print(classification_report(sentiment_val, y_pred))

In [ ]:
# # 5. เทรนและประเมินผล
# print("Training model (Might take a little longer due to FeatureUnion...)")
# pipeline.fit(text_train, sentiment_train)

# y_pred = pipeline.predict(text_val)

# print("Accuracy:", accuracy_score(sentiment_val, y_pred))
# print(classification_report(sentiment_val, y_pred))